# Hybrid retrieval over the unified Swiss legal corpus

End-to-end on a single Colab Blackwell instance. Targets **excellent recall@200** on val.csv
by fusing five complementary channels:

1. **Vector ANN** — Qwen3-Embedding-8B at fp16, full-corpus exact dot product on GPU.
2. **Statute anchor** — fixed regex parses `Art. N [Abs/Bst/Lit ...] CODE` from the query, expands law-code aliases (`StPO ↔ 312.0 ↔ CPP`), looks up `statute_links`.
3. **Case / docket anchor** — exact and `court_base` matches from `case_links`.
4. **Court-base expansion** — when any consideration of a court_base hits, pull ALL siblings (catches `BGE 137 IV 122 E. 6.2 / 6.4 / 4.2 / ...`).
5. **Adjacent-law expansion** — when a law card hits, pull next/previous article in the same statute.

Channels merged via reciprocal-rank fusion (k=60) plus a small authority-score boost,
then capped at top-K. Citation normalization (suffix-strip + SC-number canonicalization)
is applied symmetrically to gold and candidates so eval is fair.

## Data flow

```
  Drive: data/val.csv ── encode (Qwen3-Emb-8B + Instruct prefix) ──▶ Q (n_queries, 4096)
  Drive: artifacts/embeddings/qwen3_8b_unified_chunk*.npy ──▶ concat ──▶ doc_emb_gpu (2.65M, 4096) fp16
  Drive: artifacts/retrieval_metadata.zip ──▶ unzip ──▶ docs_meta + statute_links + case_links + adjacent_law_links + aliases

  per-query:
      vector       :  doc_emb_gpu @ q.T      → top-1500 by cosine
      statute      :  statute_index_norm     → docs citing aliased statute
      case         :  case_index             → docs citing target citation/court_base
      court_base_x :  for each hit's court_base, pull siblings (up to 30)
      adjacent_law :  for each law hit, pull next/prev article siblings
    ── union ── RRF(k=60, weights, authority_boost) ── top-K ──▶ candidates
    ── normalize_citation(strip suffix, alias→SC) symmetrically ──▶ recall@K vs gold
```

## Required inputs on Drive (`MyDrive/swiss_law/` by default)

| path | size | source |
|---|---|---|
| `data/val.csv` (or test/train) | small | local repo `data/` |
| `artifacts/embeddings/qwen3_8b_unified_chunk*.npy` (27 files) | ~22 GB | already there |
| `artifacts/embeddings/qwen3_8b_unified_manifest.parquet` | ~50 MB | already there |
| `artifacts/retrieval_metadata.zip` | **~220 MB** | run `python scripts/extract_retrieval_metadata.py` locally and upload |


In [1]:
# 1. Install dependencies
%pip -q install --upgrade pip
%pip -q install "sentence-transformers>=3.3" "transformers>=4.51" "accelerate>=0.34" "pyarrow>=15" pandas einops

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 33.7 MB/s eta 0:00:00


In [2]:
# 2. Mount Drive and configure paths.
import os, sys, json, time, gc, glob, math, re, warnings, zipfile
from collections import defaultdict, Counter
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

ROOT       = '/content/drive/MyDrive/swiss_law'
DATA_DIR   = f'{ROOT}/data'
EMB_DIR    = f'{ROOT}/artifacts/embeddings'
META_ZIP   = f'{ROOT}/artifacts/retrieval_metadata.zip'
META_DIR   = '/content/metadata'
OUT_DIR    = f'{ROOT}/artifacts/eval'
os.makedirs(OUT_DIR, exist_ok=True)

# Eval target — bumped to 10k so we can measure recall@1000/2000/5000/10000.
SPLIT          = 'val'
TOP_K          = 10000
QUERY_LIMIT    = 0

# Channel budgets
BUDGET_VECTOR    = 5000               # bigger pool to feed expansion
BUDGET_STATUTE   = 800
BUDGET_CASE      = 600
BUDGET_COURT_EXP = 1500
BUDGET_LAW_EXP   = 400
BUDGET_LAW_DIRECT = 400
BUDGET_SAME_CODE  = 300
# NEW
BUDGET_STATUTE_COOC = 800             # statute co-occurrence pulls related law cards
BUDGET_CITE_GRAPH   = 1500            # citation graph 1-hop expansion

# Court-base expansion seeding from vector
CB_SEED_FROM_VECTOR = 1000
CB_PER_BASE          = 30

# Citation-graph expansion seeding from union of channels
CG_SEED_FROM_VECTOR = 500
CG_PER_DOC           = 8              # neighbors per seed doc

# Fusion params
RRF_K            = 60
AUTHORITY_ALPHA  = 0.20
CHANNEL_WEIGHTS  = {
    'vector':              1.5,
    'law_card_direct':     2.5,
    'statute':             1.0,
    'case':                1.4,
    'statute_co_occurrence': 1.6,     # NEW — high precision: docs that co-cite our statute
    'citation_graph':      1.2,       # NEW — graph 1-hop neighbors
    'court_base_x':        0.9,
    'adjacent_law':        0.7,
    'same_law_code':       0.5,
}

p = torch.cuda.get_device_properties(0)
print(f'GPU: {p.name}  VRAM: {p.total_memory/1e9:.1f} GB  sm_{p.major}{p.minor}')
print(f'split: {SPLIT}  top_k: {TOP_K}')

Mounted at /content/drive
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition  VRAM: 102.0 GB  sm_120
split: val  top_k: 10000


In [3]:
# 3. Extract the metadata zip and load all small tables.
if os.path.exists(META_DIR):
    pass  # already extracted from a prior run
else:
    os.makedirs(META_DIR, exist_ok=True)
    with zipfile.ZipFile(META_ZIP) as z:
        z.extractall(META_DIR)

t0 = time.time()
docs        = pd.read_parquet(f'{META_DIR}/docs_meta.parquet')
statutes_df = pd.read_parquet(f'{META_DIR}/statute_links.parquet')
cases_df    = pd.read_parquet(f'{META_DIR}/case_links.parquet')
adj_df      = pd.read_parquet(f'{META_DIR}/adjacent_law_links.parquet')
aliases     = json.loads(open(f'{META_DIR}/law_code_aliases.json').read())
manifest    = pd.read_parquet(f'{EMB_DIR}/qwen3_8b_unified_manifest.parquet')
print(f'load time: {time.time()-t0:.1f}s')
print(f'docs        : {len(docs):,}')
print(f'statute_links: {len(statutes_df):,}')
print(f'case_links  : {len(cases_df):,}')
print(f'adj_links   : {len(adj_df):,}')
print(f'manifest    : {len(manifest):,}')
assert len(docs) == len(manifest), 'docs/manifest mismatch'

load time: 8.7s
docs        : 2,652,248
statute_links: 5,295,519
case_links  : 1,813,789
adj_links   : 347,740
manifest    : 2,652,248


In [4]:
# 4. Build the alias map and a citation-normalization function.
# v3 change: ALL_KNOWN_FORMS now includes every law_code that actually appears
# in the corpus's law family (~2,063 codes) on top of the hardcoded aliases.
# This makes anchor parsing accept any abbreviation found in laws_de.csv.

# 4a. Hardcoded canonical aliases (high-confidence cross-language groups)
ALIAS_TO_FORMS = {}
ALIAS_TO_CANONICAL = {}
ALL_KNOWN_FORMS = set()
for de_code, alts in aliases.items():
    forms = [de_code] + list(alts)
    sc = next((a for a in alts if re.match(r'^\d{3}', a)), None)
    canonical = sc or de_code
    for f in forms:
        ALIAS_TO_FORMS[f]     = forms
        ALIAS_TO_CANONICAL[f] = canonical
    ALL_KNOWN_FORMS.update(forms)

# 4b. Add every law_code from the corpus (laws_de.csv law family)
corpus_law_codes = set(c for c in docs.loc[docs['family']=='law', 'law_code'].dropna() if c)
for code in corpus_law_codes:
    if code not in ALL_KNOWN_FORMS:
        ALIAS_TO_FORMS[code]     = [code]    # standalone code with no known cross-language alias
        ALIAS_TO_CANONICAL[code] = code      # canonical = itself
ALL_KNOWN_FORMS.update(corpus_law_codes)

print(f'known law-code forms: {len(ALL_KNOWN_FORMS):,}  '
      f'(hardcoded={sum(len([de]+list(a)) for de,a in aliases.items()):,}, '
      f'+from corpus={len(corpus_law_codes):,})')

# 4c. Combined alias regex: rewrite any non-canonical form to its canonical SC number
_NON_CANONICAL = sorted(
    [f for f in ALIAS_TO_CANONICAL if f != ALIAS_TO_CANONICAL[f]],
    key=len, reverse=True,                # longer first to avoid prefix shadowing
)
ALIAS_RE = re.compile(rf'\b({"|".join(re.escape(f) for f in _NON_CANONICAL)})\b') if _NON_CANONICAL else None
STATUTE_SUFFIX_RE = re.compile(r'\s+(Abs|Bst|Lit|Ziff|Ch|Cpv|cpv)\.?\s*[\w]+', re.IGNORECASE)

def alias_replace(text: str) -> str:
    if ALIAS_RE is None:
        return text
    return ALIAS_RE.sub(lambda m: ALIAS_TO_CANONICAL[m.group(1)], text)

def normalize_citation(c, *, strip_suffix: bool = True) -> str:
    """Canonicalize a citation for fair recall@K matching.

    Rules applied symmetrically to gold and candidates:
      - whitespace collapse
      - law-code aliases -> SC number (StPO -> 312.0, CPP -> 312.0, AVS -> 831.10, ...)
      - drop Abs/Bst/Lit/Ziff suffixes (statute-side only; BGE/docket strings don't match)
    Codes that have no cross-language alias (rare cantonal codes, V-HFKG, etc.) are
    left as-is so the gold and the corpus both produce the same canonical form.
    """
    if not isinstance(c, str):
        return ''
    c = re.sub(r'\s+', ' ', c).strip()
    c = alias_replace(c)
    if strip_suffix:
        c = STATUTE_SUFFIX_RE.sub('', c)
    return c.strip()

# Smoke check across the variety of corpus patterns
print()
for s in [
    'Art. 221 Abs. 1 lit. b StPO',                  # cross-language alias case
    'Art. 221 312.0',                               # SC number form
    'Art. 221 CPP',                                 # French alias
    'Art. 38a Abs. 1 TZV',                          # article suffix (38a)
    'Art. 20c Abs. 3 VÜPF',                         # article suffix + umlaut code
    'Art. 67 Abs. 1 V-HFKG',                        # hyphenated code
    'Art. 51 Abs. 2 AsylV 2',                       # code with trailing number
    'Art. 13 Abs. 3bis 910.18',                     # bis paragraph + SC number
    'Art. 8 Abs. 1 442.132.3',                      # multi-segment SC number
    'BGE 137 IV 122 E. 6.2',
    '1B_210/2023 E. 4.1',
    '1A.204/2004 14.12.2004 E. A',                  # period-separated docket
    'U 421/00 07.05.2002 E. A',                     # old-style court
    'Art. 100 Abs. 1 BGG',
]:
    print(f'  {s:35s} -> {normalize_citation(s)}')

known law-code forms: 2,147  (hardcoded=126, +from corpus=2,063)

  Art. 221 Abs. 1 lit. b StPO         -> Art. 221 312.0
  Art. 221 312.0                      -> Art. 221 312.0
  Art. 221 CPP                        -> Art. 221 312.0
  Art. 38a Abs. 1 TZV                 -> Art. 38a TZV
  Art. 20c Abs. 3 VÜPF                -> Art. 20c VÜPF
  Art. 67 Abs. 1 V-HFKG               -> Art. 67 V-HFKG
  Art. 51 Abs. 2 AsylV 2              -> Art. 51 AsylV 2
  Art. 13 Abs. 3bis 910.18            -> Art. 13 910.18
  Art. 8 Abs. 1 442.132.3             -> Art. 8 442.132.3
  BGE 137 IV 122 E. 6.2               -> BGE 137 IV 122 E. 6.2
  1B_210/2023 E. 4.1                  -> 1B_210/2023 E. 4.1
  1A.204/2004 14.12.2004 E. A         -> 1A.204/2004 14.12.2004 E. A
  U 421/00 07.05.2002 E. A            -> U 421/00 07.05.2002 E. A
  Art. 100 Abs. 1 BGG                 -> Art. 100 173.110


In [5]:
# 5. Build inverted indexes for the anchor and expansion channels.
t0 = time.time()

row_to_docid = manifest['doc_id'].to_numpy()
docid_to_row = {d: i for i, d in enumerate(row_to_docid)}
docs_by_id   = docs.set_index('doc_id')

_cb_mask = (docs['family'] == 'court') & docs['court_base'].notna() & (docs['court_base'] != '')
court_base_groups = (
    docs[_cb_mask].groupby('court_base')['doc_id'].apply(list).to_dict()
)
print(f'  court_base groups: {len(court_base_groups):,}')

statutes_df['statute_norm'] = statutes_df['statute'].apply(normalize_citation)
statute_to_docs = statutes_df.groupby('statute_norm')['doc_id'].apply(list).to_dict()
print(f'  unique normalized statutes (court-paragraph citations): {len(statute_to_docs):,}')

doc_to_statutes = statutes_df.groupby('doc_id')['statute_norm'].apply(set).to_dict()
print(f'  doc_to_statutes: {len(doc_to_statutes):,} docs with at least one statute citation')

docs_law = docs[docs['family'] == 'law'].copy()
docs_law['cit_norm'] = docs_law['citation'].apply(normalize_citation)
law_cit_to_docs = docs_law.groupby('cit_norm')['doc_id'].apply(list).to_dict()
print(f'  unique normalized law-card citations: {len(law_cit_to_docs):,}')

docs_law_sorted = docs_law.sort_values('authority_score', ascending=False)
law_code_to_docs = docs_law_sorted.groupby('law_code')['doc_id'].apply(list).to_dict()
print(f'  law_code groups: {len(law_code_to_docs):,}')

case_to_docs = cases_df.groupby('target_citation')['doc_id'].apply(list).to_dict()
case_base_to_docs = cases_df.groupby('target_base')['doc_id'].apply(list).to_dict()
print(f'  unique target citations: {len(case_to_docs):,}  target_bases: {len(case_base_to_docs):,}')

# doc -> outgoing target_citations and target_bases (cases this doc cites)
doc_to_outgoing       = cases_df.groupby('doc_id')['target_citation'].apply(list).to_dict()
doc_to_outgoing_bases = cases_df.groupby('doc_id')['target_base'].apply(lambda s: list(set(s))).to_dict()
print(f'  doc_to_outgoing: {len(doc_to_outgoing):,} docs with outgoing case citations')

neighbor_to_docs   = adj_df.groupby('neighbor_citation')['doc_id'].apply(list).to_dict()
docid_to_neighbors = adj_df.groupby('doc_id')['neighbor_citation'].apply(list).to_dict()
print(f'  adjacent_law neighbors: {len(neighbor_to_docs):,}')

_doc_id_to_idx = {d: i for i, d in enumerate(docs['doc_id'].to_numpy())}
_citations_arr   = docs['citation'].to_numpy()
_families_arr    = docs['family'].to_numpy()
_court_bases_arr = docs['court_base'].fillna('').to_numpy()
_authority_arr   = docs['authority_score'].to_numpy()

citation_to_docid = {}
for cit, did in zip(_citations_arr, docs['doc_id'].to_numpy()):
    if cit and cit not in citation_to_docid:
        citation_to_docid[cit] = did
print(f'  citation_to_docid: {len(citation_to_docid):,}')

def doc_meta(doc_id):
    i = _doc_id_to_idx.get(doc_id)
    if i is None:
        return ('', '', '', 0.0)
    return (_citations_arr[i], _families_arr[i], _court_bases_arr[i], float(_authority_arr[i] or 0.0))

print(f'index build: {time.time()-t0:.1f}s')

  court_base groups: 178,593
  unique normalized statutes (court-paragraph citations): 171,498
  doc_to_statutes: 1,414,972 docs with at least one statute citation
  unique normalized law-card citations: 70,935
  law_code groups: 2,063
  unique target citations: 283,108  target_bases: 125,795
  doc_to_outgoing: 744,046 docs with outgoing case citations
  adjacent_law neighbors: 175,930
  citation_to_docid: 2,161,111
index build: 53.0s


In [6]:
# 6. Load all 27 doc embedding chunks into a single GPU tensor.
# Total: 2,652,248 * 4096 * 2 bytes = ~21.7 GB; comfortable in 95 GB VRAM.
DIM = 4096
chunk_paths = sorted(glob.glob(f'{EMB_DIR}/qwen3_8b_unified_chunk*.npy'))
print(f'discovered {len(chunk_paths)} chunk file(s)')

total_rows = sum(np.load(p, mmap_mode='r').shape[0] for p in chunk_paths)
print(f'total rows across chunks: {total_rows:,}  (manifest has {len(manifest):,})')
assert total_rows == len(manifest), 'chunk row count != manifest row count'

t0 = time.time()
doc_emb_gpu = torch.empty((total_rows, DIM), dtype=torch.float16, device='cuda')
offset = 0
for path in chunk_paths:
    arr = np.load(path)
    n = arr.shape[0]
    doc_emb_gpu[offset:offset+n] = torch.from_numpy(arr).cuda()
    offset += n
    print(f'  {os.path.basename(path):40s}  rows={n:>6,}  cumulative={offset:>10,}', flush=True)
    del arr

print(f'\nload time: {time.time()-t0:.1f}s')
print(f'doc_emb_gpu: {doc_emb_gpu.shape} {doc_emb_gpu.dtype}')
print(f'VRAM allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB')

discovered 27 chunk file(s)
total rows across chunks: 2,652,248  (manifest has 2,652,248)
  qwen3_8b_unified_chunk000.npy             rows=100,000  cumulative=   100,000
  qwen3_8b_unified_chunk001.npy             rows=100,000  cumulative=   200,000
  qwen3_8b_unified_chunk002.npy             rows=100,000  cumulative=   300,000
  qwen3_8b_unified_chunk003.npy             rows=100,000  cumulative=   400,000
  qwen3_8b_unified_chunk004.npy             rows=100,000  cumulative=   500,000
  qwen3_8b_unified_chunk005.npy             rows=100,000  cumulative=   600,000
  qwen3_8b_unified_chunk006.npy             rows=100,000  cumulative=   700,000
  qwen3_8b_unified_chunk007.npy             rows=100,000  cumulative=   800,000
  qwen3_8b_unified_chunk008.npy             rows=100,000  cumulative=   900,000
  qwen3_8b_unified_chunk009.npy             rows=100,000  cumulative= 1,000,000
  qwen3_8b_unified_chunk010.npy             rows=100,000  cumulative= 1,100,000
  qwen3_8b_unified_chunk011.np

In [7]:
# 7. Anchor parsing for the query side.
# v3 changes: regex coverage matches the full set of patterns observed in the
# corpus (175k law citations + 1.99M court citations).
#
#   Article number       \d+[a-z]*                 -> 38a, 20c, 97a
#   Suffix value         [\w]+                     -> b, 3bis, a, 1
#   Law-code             SC-number | umlaut-aware abbrev | abbrev+num | hyphenated
#   Old-style court      \b[A-Z]\s+\d{1,4}/\d{2,4} -> U 421/00, I 89/02, H 418/99
#   Docket separator     [_.]                      -> 9C_663/2011 OR 1A.204/2004
#   Suffix keywords      case-insensitive (?i:...) -> 'lit. b' matches but 'lit' doesn't
#                                                     qualify as a law code
import unicodedata

def normalize_query_text(t: str) -> str:
    t = unicodedata.normalize('NFKC', str(t))
    t = t.replace('‑', '-').replace('–', '-').replace('—', '-')
    return t

# Article-pattern: covers all observed forms in laws_de.csv
ART_PATTERN = re.compile(
    r'\bArt\.?\s*(\d+[a-z]*)'                                       # article num + optional letter suffix
    r'((?:\s+(?i:Abs|Bst|Lit|Ziff|Ch|Cpv)\.?\s*[\w]+)*)'            # optional Abs/Bst/Lit etc. (allow 3bis, lit. b)
    r'\s+'
    r'('
        r'\d{1,4}(?:\.\d+)+'                                        # SC number with at least one dot (131.211, 442.132.3)
        r'|[A-Za-zÀ-ÿ][\wÀ-ÿ]*(?:-[A-Za-z][\wÀ-ÿ]*)*(?:\s+\d+)?'    # abbreviation: umlaut-aware, hyphenated, optional trailing number
    r')'
)
# BGE: full + base + page form (BGE 139 I 2 S. 7 from val_001 normalization output)
BGE_FULL_RE = re.compile(r'\bBGE\s+\d{3}\s+[IVX]{1,4}\s+\d+[a-z]?(?:\s+E\.\s*[\d\.]+)?\b')
BGE_BASE_RE = re.compile(r'\bBGE\s+\d{3}\s+[IVX]{1,4}\s+\d+[a-z]?\b')
# Modern docket: 53 known prefixes (1B, 1C, 2C, 4A, 6B, 9C, 12T, 13Y...) -> [A-Z]+ at length 1-4
DOCKET_RE     = re.compile(r'\b\d{1,2}[A-Z]+[._]\d{1,5}/\d{2,4}\b')
# Old-style court: pre-2007 single-letter prefixes (U, I, C, H, K, B, P, M, E, F)
OLD_COURT_RE  = re.compile(r'\b[A-Z]\s+\d{1,4}/\d{2,4}\b')

def parse_anchors(query: str) -> dict:
    """Extract deterministic citation anchors from a free-text query."""
    query = normalize_query_text(query)
    cases = sorted(set(BGE_FULL_RE.findall(query)))
    case_bases = sorted(set(BGE_BASE_RE.findall(query)))
    dockets = sorted(set(DOCKET_RE.findall(query)))
    # Old-style courts are single-letter so be careful: only accept if followed by /year
    old = sorted(set(OLD_COURT_RE.findall(query)))
    statutes = []
    for m in ART_PATTERN.finditer(query):
        art_num = m.group(1)
        suffix = (m.group(2) or '').strip()
        code = m.group(3).strip()
        # Reject if code unknown — avoids 'lit' from suffix bleeding into the law-code group
        if code not in ALL_KNOWN_FORMS:
            continue
        forms = ALIAS_TO_FORMS.get(code, [code])
        canonical_code = ALIAS_TO_CANONICAL.get(code, code)
        statutes.append({
            'article':    art_num,
            'code':       code,
            'canonical':  canonical_code,
            'forms':      forms,
            'suffix_raw': suffix,
            'normalized': f'Art. {art_num} {canonical_code}',
        })
    return {
        'cases':       cases,
        'case_bases':  case_bases,
        'dockets':     list(dockets) + list(old),     # treat old-style as docket for case_anchor
        'statutes':    statutes,
    }

# Smoke checks
demo_qs = [
    # val_001-style with non-breaking hyphens
    ('val_001 query stub',
     'May a court order extension of pre‑trial detention under Art. 221 Abs. 1 lit. b StPO '
     'consistent with proportionality? See BGE 137 IV 122 E. 6.2 and 1B_210/2023 E. 4.1, '
     'also Art. 100 Abs. 1 BGG.'),
    # Old-style court mention
    ('old-style stub',
     'In U 421/00 the Federal Court held... see also I 89/02 E. 5 and H 418/99.'),
    # Hyphenated and umlaut codes
    ('rare codes stub',
     'The decision applies Art. 67 Abs. 1 V-HFKG. See also Art. 20c Abs. 3 VÜPF and '
     'Art. 51 Abs. 2 AsylV 2.'),
    # Period-separated docket
    ('period docket stub',
     'In 1A.204/2004 the court considered Art. 8 Abs. 2 442.132.3.'),
]
for label, q in demo_qs:
    print(f'\n[{label}]')
    a = parse_anchors(q)
    print(f'  statutes: {[s["normalized"] for s in a["statutes"]]}')
    print(f'  cases: {a["cases"][:3]}')
    print(f'  case_bases: {a["case_bases"][:3]}')
    print(f'  dockets: {a["dockets"][:5]}')


[val_001 query stub]
  statutes: ['Art. 221 312.0', 'Art. 100 173.110']
  cases: ['BGE 137 IV 122 E. 6.2']
  case_bases: ['BGE 137 IV 122']
  dockets: ['1B_210/2023']

[old-style stub]
  statutes: []
  cases: []
  case_bases: []
  dockets: ['H 418/99', 'I 89/02', 'U 421/00']

[rare codes stub]
  statutes: ['Art. 67 V-HFKG', 'Art. 20c VÜPF', 'Art. 51 AsylV 2']
  cases: []
  case_bases: []
  dockets: []

[period docket stub]
  statutes: ['Art. 8 442.132.3']
  cases: []
  case_bases: []
  dockets: ['1A.204/2004']


In [8]:
# 8. Channel implementations.

def vector_search(q_np: np.ndarray, k: int = BUDGET_VECTOR) -> list[dict]:
    q = torch.from_numpy(q_np.astype(np.float32)).cuda().half()
    n = q.norm()
    if n == 0:
        return []
    q = q / n
    scores = (doc_emb_gpu @ q.unsqueeze(-1)).squeeze(-1).float()
    top_scores, top_idx = torch.topk(scores, k)
    top_idx = top_idx.cpu().numpy()
    top_scores = top_scores.cpu().numpy()
    out = []
    for ri, sc in zip(top_idx, top_scores):
        d = row_to_docid[int(ri)]
        cit, fam, base, auth = doc_meta(d)
        out.append({'doc_id': d, 'citation': cit, 'family': fam, 'court_base': base,
                    'authority': auth, 'channel': 'vector', 'raw': float(sc)})
    return out


def law_card_direct(parsed_statutes, k=BUDGET_LAW_DIRECT):
    if not parsed_statutes:
        return []
    seen = set(); out = []
    for st in parsed_statutes:
        target = normalize_citation(f'Art. {st["article"]} {st["canonical"]}')
        for d in law_cit_to_docs.get(target, ())[:k]:
            if d in seen: continue
            seen.add(d)
            cit, fam, base, auth = doc_meta(d)
            out.append({'doc_id': d, 'citation': cit, 'family': fam, 'court_base': base,
                        'authority': auth, 'channel': 'law_card_direct', 'raw': auth, 'anchor': target})
    out.sort(key=lambda x: x['raw'], reverse=True)
    return out[:k]


def same_law_code(parsed_statutes, k=BUDGET_SAME_CODE):
    if not parsed_statutes:
        return []
    codes_seen = set(); out = []
    per_code = max(1, k // max(len(parsed_statutes), 1))
    for st in parsed_statutes:
        for code in [st['canonical']] + list(st['forms']):
            if code in codes_seen: break
            if code in law_code_to_docs:
                codes_seen.add(code)
                for d in law_code_to_docs[code][:per_code]:
                    cit, fam, base, auth = doc_meta(d)
                    out.append({'doc_id': d, 'citation': cit, 'family': fam, 'court_base': base,
                                'authority': auth, 'channel': 'same_law_code', 'raw': auth, 'anchor': code})
                break
    seen = set(); deduped = []
    for h in out:
        if h['doc_id'] in seen: continue
        seen.add(h['doc_id']); deduped.append(h)
    return deduped[:k]


def statute_anchor(parsed_statutes, k=BUDGET_STATUTE):
    if not parsed_statutes:
        return []
    seen = set(); out = []
    for st in parsed_statutes:
        for code in st['forms']:
            key = normalize_citation(f'Art. {st["article"]} {code}')
            for d in statute_to_docs.get(key, ())[:k]:
                if d in seen: continue
                seen.add(d)
                cit, fam, base, auth = doc_meta(d)
                out.append({'doc_id': d, 'citation': cit, 'family': fam, 'court_base': base,
                            'authority': auth, 'channel': 'statute', 'raw': auth, 'anchor': st['normalized']})
    out.sort(key=lambda x: x['raw'], reverse=True)
    return out[:k]


def case_anchor(parsed_cases, parsed_bases, parsed_dockets, k=BUDGET_CASE):
    seen = set(); out = []
    for c in parsed_cases:
        for d in case_to_docs.get(c, ())[:k]:
            if d in seen: continue
            seen.add(d)
            cit, fam, base, auth = doc_meta(d)
            out.append({'doc_id': d, 'citation': cit, 'family': fam, 'court_base': base,
                        'authority': auth, 'channel': 'case', 'raw': auth, 'anchor': c})
    for b in list(parsed_bases) + list(parsed_dockets):
        for d in case_base_to_docs.get(b, ())[:k]:
            if d in seen: continue
            seen.add(d)
            cit, fam, base, auth = doc_meta(d)
            out.append({'doc_id': d, 'citation': cit, 'family': fam, 'court_base': base,
                        'authority': auth, 'channel': 'case', 'raw': auth, 'anchor': b})
    for c in parsed_cases + parsed_bases + parsed_dockets:
        ri_arr = np.where(_citations_arr == c)[0][:k]
        for ri in ri_arr:
            d = docs['doc_id'].iloc[int(ri)]
            if d in seen: continue
            seen.add(d)
            cit, fam, base, auth = doc_meta(d)
            out.append({'doc_id': d, 'citation': cit, 'family': fam, 'court_base': base,
                        'authority': auth, 'channel': 'case', 'raw': auth, 'anchor': c})
    out.sort(key=lambda x: x['raw'], reverse=True)
    return out[:k]


def statute_co_occurrence(parsed_statutes, k=BUDGET_STATUTE_COOC, max_citing_docs=10000, top_cooc=300):
    """Procedural-scaffolding via corpus statistics: docs citing Q tend to also cite Q's
    procedural neighbors (Art. 100 BGG for federal appeals, Art. 422/428 StPO for costs, ...).
    Pull law cards for the top co-occurring statutes."""
    if not parsed_statutes:
        return []
    target_norms = set()
    for st in parsed_statutes:
        for code in st['forms']:
            target_norms.add(normalize_citation(f'Art. {st["article"]} {code}'))
    citing_docs = set()
    for t in target_norms:
        for d in statute_to_docs.get(t, [])[:max_citing_docs]:
            citing_docs.add(d)
    if not citing_docs:
        return []
    cooc = Counter()
    for d in citing_docs:
        for s in doc_to_statutes.get(d, set()) - target_norms:
            cooc[s] += 1
    seen = set(); out = []
    for s, n in cooc.most_common(top_cooc):
        for d in law_cit_to_docs.get(s, [])[:3]:
            if d in seen: continue
            seen.add(d)
            cit, fam, base, auth = doc_meta(d)
            out.append({'doc_id': d, 'citation': cit, 'family': fam, 'court_base': base,
                        'authority': auth, 'channel': 'statute_co_occurrence',
                        'raw': float(auth) + math.log1p(n)/10.0,
                        'anchor': f'cooc(n={n}):{s}'})
            if len(out) >= k:
                return out
    return out[:k]


def citation_graph_expansion(seed_hits, k=BUDGET_CITE_GRAPH, per_doc=CG_PER_DOC,
                             seed_top=CG_SEED_FROM_VECTOR, max_bases=300, per_base=10):
    """1-hop citation-graph expansion via court_base.

    Steps:
      1. From the top of the seed pool, collect each seed court doc's
            (a) its OWN court_base (so that "any case citing this base" gets pulled in)
            (b) its OUTGOING target_bases (the bases it cites — extends 1 hop)
      2. For each unique target base, pull docs that cite ANY consideration of that base
         via case_base_to_docs.
    For BGE 137 IV 122 in seed → catches the 360 docs citing it AND BGE 132 I 21
    (which BGE 137 IV 122 cites) → 526 more docs.
    """
    if not seed_hits:
        return []
    seen_docs = {h['doc_id'] for h in seed_hits}
    target_bases = []
    target_bases_set = set()

    for h in seed_hits[:seed_top]:
        if h.get('family') != 'court':
            continue
        b = h.get('court_base')
        if b and b not in target_bases_set:
            target_bases_set.add(b); target_bases.append(b)
        for t_base in doc_to_outgoing_bases.get(h['doc_id'], ())[:per_doc]:
            if t_base and t_base not in target_bases_set:
                target_bases_set.add(t_base); target_bases.append(t_base)
        if len(target_bases) >= max_bases:
            break

    out = []
    for base in target_bases:
        for d in case_base_to_docs.get(base, ())[:per_base]:
            if d in seen_docs: continue
            seen_docs.add(d)
            cit, fam, b, auth = doc_meta(d)
            out.append({'doc_id': d, 'citation': cit, 'family': fam, 'court_base': b,
                        'authority': auth, 'channel': 'citation_graph', 'raw': auth,
                        'anchor': f'cited_base({base})'})
            if len(out) >= k:
                return out
        # Also pull the cited base's own considerations (so BGE 132 I 21 considerations
        # are in the pool when BGE 137 IV 122 cites BGE 132 I 21).
        for d in court_base_groups.get(base, ())[:per_base]:
            if d in seen_docs: continue
            seen_docs.add(d)
            cit, fam, b, auth = doc_meta(d)
            out.append({'doc_id': d, 'citation': cit, 'family': fam, 'court_base': b,
                        'authority': auth, 'channel': 'citation_graph', 'raw': auth,
                        'anchor': f'in_base({base})'})
            if len(out) >= k:
                return out
    return out[:k]


def expand_court_base(seed_hits, k_per_base=CB_PER_BASE, max_total=BUDGET_COURT_EXP):
    seen_bases = []; seen_b_set = set()
    for h in seed_hits:
        b = h.get('court_base')
        if b and b not in seen_b_set:
            seen_b_set.add(b); seen_bases.append(b)
    seen_docs = {h['doc_id'] for h in seed_hits}
    out = []
    for base in seen_bases:
        for d in court_base_groups.get(base, ())[:k_per_base]:
            if d in seen_docs: continue
            seen_docs.add(d)
            cit, fam, b, auth = doc_meta(d)
            out.append({'doc_id': d, 'citation': cit, 'family': fam, 'court_base': b,
                        'authority': auth, 'channel': 'court_base_x', 'raw': auth, 'anchor': base})
        if len(out) >= max_total:
            break
    out.sort(key=lambda x: x['raw'], reverse=True)
    return out[:max_total]


def expand_adjacent_law(seed_hits, max_total=BUDGET_LAW_EXP):
    out = []
    seen_docs = {h['doc_id'] for h in seed_hits}
    for h in seed_hits:
        if h.get('family') != 'law':
            continue
        for n_cit in docid_to_neighbors.get(h['doc_id'], ())[:6]:
            for d in neighbor_to_docs.get(n_cit, ())[:5]:
                if d == h['doc_id'] or d in seen_docs: continue
                seen_docs.add(d)
                cit2, fam, base, auth = doc_meta(d)
                out.append({'doc_id': d, 'citation': cit2, 'family': fam, 'court_base': base,
                            'authority': auth, 'channel': 'adjacent_law', 'raw': auth,
                            'anchor': h['citation']})
                if len(out) >= max_total:
                    return out
    return out

print('channels: vector, law_card_direct, same_law_code, statute, case, '
      'statute_co_occurrence (NEW), citation_graph (NEW v2 — by court_base), court_base_x, adjacent_law')

channels: vector, law_card_direct, same_law_code, statute, case, statute_co_occurrence (NEW), citation_graph (NEW v2 — by court_base), court_base_x, adjacent_law


In [9]:
# 9. Hybrid retrieve function: fuse all channels via reciprocal-rank fusion (RRF).
def reciprocal_rank_fusion(channel_results, *, k_const, weights, alpha):
    scores = defaultdict(float); meta = {}; per_channel_rank = defaultdict(dict)
    for channel, results in channel_results.items():
        w = weights.get(channel, 1.0)
        for rank, item in enumerate(results, start=1):
            d = item['doc_id']
            scores[d] += w / (k_const + rank)
            per_channel_rank[channel][d] = rank
            if d not in meta:
                meta[d] = {'doc_id': d, 'citation': item.get('citation'),
                          'family': item.get('family'),
                          'court_base': item.get('court_base'),
                          'authority': item.get('authority', 0.0)}
    out = []
    for d, s in scores.items():
        e = dict(meta[d])
        e['fused_raw'] = s
        e['fused_score'] = s * (1.0 + alpha * (e.get('authority') or 0.0))
        e['channel_ranks'] = {c: per_channel_rank[c][d] for c in per_channel_rank if d in per_channel_rank[c]}
        out.append(e)
    out.sort(key=lambda r: r['fused_score'], reverse=True)
    return out


def retrieve(query, q_emb, top_k=TOP_K):
    anchors = parse_anchors(query)
    timings = {}

    t0 = time.time()
    v = vector_search(q_emb, k=BUDGET_VECTOR)
    timings['vector'] = time.time() - t0

    t0 = time.time()
    lcd = law_card_direct(anchors['statutes'], k=BUDGET_LAW_DIRECT) if anchors['statutes'] else []
    timings['law_card_direct'] = time.time() - t0

    t0 = time.time()
    slc = same_law_code(anchors['statutes'], k=BUDGET_SAME_CODE) if anchors['statutes'] else []
    timings['same_law_code'] = time.time() - t0

    t0 = time.time()
    s = statute_anchor(anchors['statutes'], k=BUDGET_STATUTE) if anchors['statutes'] else []
    timings['statute'] = time.time() - t0

    t0 = time.time()
    c = case_anchor(anchors['cases'], anchors['case_bases'], anchors['dockets'],
                    k=BUDGET_CASE) if (anchors['cases'] or anchors['case_bases'] or anchors['dockets']) else []
    timings['case'] = time.time() - t0

    # NEW: statute co-occurrence
    t0 = time.time()
    sco = statute_co_occurrence(anchors['statutes'], k=BUDGET_STATUTE_COOC) if anchors['statutes'] else []
    timings['statute_co_occurrence'] = time.time() - t0

    # Court-base expansion seeded from vector + statute + case + lcd
    t0 = time.time()
    seed_for_expansion = (v[:CB_SEED_FROM_VECTOR] + s[:80] + c[:60] + lcd[:60])
    cb = expand_court_base(seed_for_expansion, k_per_base=CB_PER_BASE, max_total=BUDGET_COURT_EXP)
    timings['court_base_x'] = time.time() - t0

    # NEW: citation graph 1-hop expansion (uses the SAME seed pool)
    t0 = time.time()
    cg = citation_graph_expansion(seed_for_expansion, k=BUDGET_CITE_GRAPH,
                                  per_doc=CG_PER_DOC, seed_top=CG_SEED_FROM_VECTOR)
    timings['citation_graph'] = time.time() - t0

    t0 = time.time()
    adj = expand_adjacent_law(seed_for_expansion, max_total=BUDGET_LAW_EXP)
    timings['adjacent_law'] = time.time() - t0

    channels = {
        'vector': v,
        'law_card_direct': lcd,
        'same_law_code': slc,
        'statute': s,
        'case': c,
        'statute_co_occurrence': sco,
        'citation_graph': cg,
        'court_base_x': cb,
        'adjacent_law': adj,
    }
    active = {k: v for k, v in channels.items() if v}

    fused = reciprocal_rank_fusion(
        active, k_const=RRF_K, weights=CHANNEL_WEIGHTS, alpha=AUTHORITY_ALPHA,
    )[:top_k]

    return {
        'query': query,
        'anchors': anchors,
        'channel_counts': {k: len(v) for k, v in channels.items()},
        'channel_timings_s': timings,
        'candidates': fused,
    }

print('retrieve() v4 ready: 9 channels, top_k=', TOP_K)

retrieve() v4 ready: 9 channels, top_k= 10000


In [10]:
# 10. Load Qwen3-Embedding-8B and encode all queries for the chosen split.
from sentence_transformers import SentenceTransformer

QWEN_INSTRUCT = (
    'Instruct: Given an English-language legal question or scenario about Swiss federal law, '
    'retrieve the Swiss statute articles or federal court decision considerations that are most '
    'directly relevant to answering it.\nQuery: '
)

csv_path = f'{DATA_DIR}/{SPLIT}.csv'
df = pd.read_csv(csv_path)
if QUERY_LIMIT:
    df = df.head(QUERY_LIMIT).reset_index(drop=True)
print(f'{SPLIT}: {len(df):,} queries  cols: {list(df.columns)}')

free_vram_for_model = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9
print(f'free VRAM for model load: ~{free_vram_for_model:.1f} GB')

t0 = time.time()
model = SentenceTransformer(
    'Qwen/Qwen3-Embedding-8B', device='cuda',
    model_kwargs={'torch_dtype': torch.bfloat16, 'attn_implementation': 'sdpa'},
    tokenizer_kwargs={'padding_side': 'left'},
)
model.max_seq_length = 768
model.eval()
print(f'model load: {time.time()-t0:.1f}s  VRAM allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB')

queries = [QWEN_INSTRUCT + str(q) for q in df['query'].tolist()]
t0 = time.time()
with torch.inference_mode():
    Q = model.encode(
        queries,
        batch_size=8,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    ).astype(np.float32)
print(f'encoded {len(Q)} queries in {time.time()-t0:.1f}s  shape={Q.shape}')

# Save for offline reuse
np.save(f'{EMB_DIR}/qwen3_8b_query_{SPLIT}.npy', Q)
print(f'saved query embeddings to {EMB_DIR}/qwen3_8b_query_{SPLIT}.npy')

# Optional: free model VRAM since vector search uses doc_emb_gpu only
del model
gc.collect()
torch.cuda.empty_cache()
print(f'VRAM after model free: {torch.cuda.memory_allocated()/1e9:.2f} GB')

val: 10 queries  cols: ['query_id', 'query', 'gold_citations']
free VRAM for model load: ~80.2 GB


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/729 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

model load: 54.9s  VRAM allocated: 36.86 GB


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

encoded 10 queries in 5.4s  shape=(10, 4096)
saved query embeddings to /content/drive/MyDrive/swiss_law/artifacts/embeddings/qwen3_8b_query_val.npy
VRAM after model free: 21.74 GB


In [11]:
# 11. Run hybrid retrieval over every query and compute recall@K.
def parse_gold(s):
    if not isinstance(s, str): return []
    return [g.strip() for g in s.split(';') if g.strip()]

def recall_at(candidates_normed, gold_normed_set, k):
    if not gold_normed_set: return 1.0
    cand_set = set(candidates_normed[:k])
    return len(cand_set & gold_normed_set) / len(gold_normed_set)

RECALL_AT = (50, 100, 200, 500, 1000, 2000, 5000, 10000)

results = []
dropped_rows = []
channel_count_sums = Counter()
channel_timing_sums = Counter()
t_total = time.time()

for i, row in df.iterrows():
    qid = row['query_id']
    query = row['query']
    gold_raw = parse_gold(row.get('gold_citations', ''))
    gold_normed_set = set(normalize_citation(g) for g in gold_raw)
    n_gold_unique = len(gold_normed_set)

    r = retrieve(query, Q[i], top_k=TOP_K)
    cand_norm = [normalize_citation(c['citation']) for c in r['candidates']]

    recalls = {f'r@{k}': recall_at(cand_norm, gold_normed_set, k) for k in RECALL_AT}
    for k, v in r['channel_counts'].items():
        channel_count_sums[k] += v
    for k, v in r['channel_timings_s'].items():
        channel_timing_sums[k] += v

    cand_set = set(cand_norm)
    found = gold_normed_set & cand_set
    dropped = gold_normed_set - cand_set
    for g in dropped:
        dropped_rows.append({'query_id': qid, 'gold_citation': g})

    rec = {
        'query_id': qid,
        'gold_total': len(gold_raw),
        'gold_unique': n_gold_unique,
        'found_unique': len(found),
        **recalls,
        'anchors': r['anchors'],
        'channel_counts': r['channel_counts'],
        'channel_timings_s': r['channel_timings_s'],
        'candidates': r['candidates'],
    }
    results.append(rec)
    if (i + 1) % 5 == 0 or (i + 1) == len(df):
        means = {k: np.mean([res[f'r@{k}'] for res in results]) for k in RECALL_AT}
        print(f'  {i+1}/{len(df)}  '
              f'r@200={means[200]:.4f}  r@1000={means[1000]:.4f}  r@10000={means[10000]:.4f}  '
              f'gold_uniq={n_gold_unique} found={len(found)}  '
              f'time/q={(time.time()-t_total) / (i+1):.2f}s', flush=True)

elapsed = time.time() - t_total
summary = {
    'split': SPLIT, 'n_queries': len(df), 'top_k': TOP_K,
    'elapsed_s': round(elapsed, 1),
    'recall_at_k': {f'r@{k}': float(np.mean([r[f"r@{k}"] for r in results])) for k in RECALL_AT},
    'channel_counts_total': dict(channel_count_sums),
    'channel_avg_time_ms': {k: int(channel_timing_sums[k] * 1000 / len(df)) for k in channel_timing_sums},
    'config': {
        'budgets': {
            'vector': BUDGET_VECTOR, 'statute': BUDGET_STATUTE, 'case': BUDGET_CASE,
            'court_base_x': BUDGET_COURT_EXP, 'adjacent_law': BUDGET_LAW_EXP,
            'law_card_direct': BUDGET_LAW_DIRECT, 'same_law_code': BUDGET_SAME_CODE,
            'statute_co_occurrence': BUDGET_STATUTE_COOC, 'citation_graph': BUDGET_CITE_GRAPH,
        },
        'rrf_k': RRF_K, 'authority_alpha': AUTHORITY_ALPHA,
        'channel_weights': CHANNEL_WEIGHTS,
        'cb_seed_from_vector': CB_SEED_FROM_VECTOR,
        'cg_seed_from_vector': CG_SEED_FROM_VECTOR,
    },
}
print('\n=== summary ===')
print(json.dumps(summary, indent=2))

out_summary = f'{OUT_DIR}/{SPLIT}_summary.json'
out_jsonl   = f'{OUT_DIR}/{SPLIT}_candidate_sets.jsonl'
out_dropped = f'{OUT_DIR}/{SPLIT}_dropped_gold.csv'
with open(out_summary, 'w') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)
# Note: candidate_sets.jsonl gets large with top_k=10k. Cap stored candidates per query at 1000.
with open(out_jsonl, 'w', encoding='utf-8') as f:
    for r in results:
        f.write(json.dumps({
            'query_id': r['query_id'],
            'gold_total': r['gold_total'],
            'gold_unique': r['gold_unique'],
            'found_unique': r['found_unique'],
            **{k: r[k] for k in r if k.startswith('r@')},
            'anchors': r['anchors'],
            'channel_counts': r['channel_counts'],
            'channel_timings_s': r['channel_timings_s'],
            'candidates': [{
                'rank': j + 1,
                'doc_id': c['doc_id'],
                'family': c['family'],
                'citation': c['citation'],
                'court_base': c['court_base'],
                'fused_score': c['fused_score'],
                'channel_ranks': c['channel_ranks'],
            } for j, c in enumerate(r['candidates'][:1000])],
        }, ensure_ascii=False) + '\n')
pd.DataFrame(dropped_rows).to_csv(out_dropped, index=False)
print(f'\nwrote {out_summary}')
print(f'wrote {out_jsonl}')
print(f'wrote {out_dropped}')

  5/10  r@200=0.2327  r@1000=0.4096  r@10000=0.5693  gold_uniq=10 found=8  time/q=0.09s
  10/10  r@200=0.2535  r@1000=0.4013  r@10000=0.5307  gold_uniq=23 found=10  time/q=0.08s

=== summary ===
{
  "split": "val",
  "n_queries": 10,
  "top_k": 10000,
  "elapsed_s": 0.8,
  "recall_at_k": {
    "r@50": 0.16956798103856927,
    "r@100": 0.20380017112753684,
    "r@200": 0.2534912984145721,
    "r@500": 0.3673237901626648,
    "r@1000": 0.4013426329283106,
    "r@2000": 0.44681271839839615,
    "r@5000": 0.5116862620187429,
    "r@10000": 0.5306882730795774
  },
  "channel_counts_total": {
    "vector": 50000,
    "law_card_direct": 18,
    "same_law_code": 850,
    "statute": 2147,
    "case": 0,
    "statute_co_occurrence": 1466,
    "citation_graph": 15000,
    "court_base_x": 15000,
    "adjacent_law": 631
  },
  "channel_avg_time_ms": {
    "vector": 39,
    "law_card_direct": 0,
    "same_law_code": 0,
    "statute": 0,
    "case": 0,
    "statute_co_occurrence": 1,
    "court_base_

In [12]:
# 12. Per-query inspection: which queries have low recall, what gold did we miss.
df_per_query = pd.DataFrame([{
    'query_id':    r['query_id'],
    'gold_total':  r['gold_total'],
    'gold_unique': r['gold_unique'],
    'found':       r['found_unique'],
    'r@200':       r['r@200'],
    'r@1000':      r['r@1000'],
    'r@10000':     r['r@10000'],
    'n_statutes_parsed': len(r['anchors']['statutes']),
    'n_cases_parsed':    len(r['anchors']['cases']) + len(r['anchors']['case_bases']) + len(r['anchors']['dockets']),
} for r in results])
print('=== per-query (sorted worst recall@200 first) ===')
print(df_per_query.sort_values('r@200').to_string(index=False))

print('\n=== top of dropped-gold (most-frequently missed citations across queries) ===')
if dropped_rows:
    drop_df = pd.DataFrame(dropped_rows)
    print(drop_df.gold_citation.value_counts().head(25).to_string())

print(f'\n=== aggregate recall ===')
for k in RECALL_AT:
    print(f'  recall@{k:>5}  =  {summary["recall_at_k"][f"r@{k}"]:.4f}')

=== per-query (sorted worst recall@200 first) ===
query_id  gold_total  gold_unique  found    r@200   r@1000  r@10000  n_statutes_parsed  n_cases_parsed
 val_003          47           45      9 0.022222 0.066667 0.200000                  0               0
 val_001          42           39     32 0.102564 0.384615 0.820513                  1               0
 val_008          29           28      3 0.107143 0.107143 0.107143                  0               0
 val_010          25           23     10 0.130435 0.304348 0.434783                  0               0
 val_009          14           12      6 0.166667 0.416667 0.500000                  0               0
 val_002          36           34     16 0.294118 0.441176 0.470588                  1               0
 val_005          11           10      8 0.300000 0.600000 0.800000                  0               0
 val_007          19           17      9 0.411765 0.470588 0.529412                  2               0
 val_004          10   